# Lab Exercise: Docker Containerization for AI Models
## AIAT 125 — Unit 4: Containerization and Orchestration

**Learning objectives**
1. Generate valid Dockerfile content programmatically and validate its structure.
2. Parse Dockerfile instructions and verify best practices.
3. Simulate container isolation — two containers with different configs on the same host.
4. Build a complete set of deployment files for containerizing an iris classifier.

**Why this matters**  
Without containerization, a model that works on your laptop can crash on a cloud server because of a different library version. Docker packages the model, its code, and all dependencies into a single isolated container — solving the "it works on my machine" problem permanently.

**Grading** — 100 points total
| Task | Points |
|---|---|
| Task 1: Dockerfile generator | 30 |
| Task 2: Dockerfile parser and validator | 25 |
| Task 3: Container isolation simulation | 25 |
| Task 4: Complete deployment file bundle | 20 |

> This lab uses Python only — no Docker installation required.


In [ ]:
# WHAT: create the working directory all four tasks write their files into.
# WHY: Task 4's final gate checks files exist in EXERCISE_DIR — everything you
# generate in this lab lands there.
import os, json, textwrap

EXERCISE_DIR = "/tmp/docker_exercise"
os.makedirs(EXERCISE_DIR, exist_ok=True)
print("Setup complete.")

---
## Background: The Dockerfile

A Dockerfile is a recipe for building a container image. The four steps from the Unit 4 curriculum:

| Step | Instruction | Purpose |
|---|---|---|
| 1 | `FROM python:3.10-slim` | Start with a base OS image that has Python |
| 2 | `RUN pip install -r requirements.txt` | Install exact library versions |
| 3 | `COPY model.joblib /app/` | Copy model weights into the container |
| 4 | `CMD ["python", "app.py"]` | Define the command that starts the service |

The `WORKDIR` instruction sets the working directory inside the container. The `EXPOSE` instruction documents which port the service listens on.

**Best practices:**
- Use `python:3.10-slim` (not `python:3.10`) to reduce image size by ~700 MB
- Copy `requirements.txt` before code so Docker caches the pip install layer
- Never embed secrets (API keys, passwords) in the Dockerfile
- Use `EXPOSE` to document the port — it's documentation, not a firewall rule

---
## Task 1 — Dockerfile Generator (30 points)

Implement `generate_dockerfile(model_filename, port, python_version, extra_packages)` that returns a valid Dockerfile string.

The Dockerfile must contain these instructions in this order:
1. `FROM python:{python_version}-slim`
2. `WORKDIR /app`
3. `COPY requirements.txt .`
4. `RUN pip install --no-cache-dir -r requirements.txt`
5. `COPY {model_filename} /app/`
6. `COPY app.py /app/`
7. `EXPOSE {port}`
8. `CMD ["python", "app.py"]`

If `extra_packages` is provided (a non-empty list), add an additional `RUN pip install {' '.join(extra_packages)}` instruction before the COPY instructions.

In [ ]:
# WHAT: Task 1 — implement generate_dockerfile(): parameters in, a complete
# Dockerfile string out.
# WHY: writing the generator forces you to know each instruction's job (FROM,
# WORKDIR, COPY, RUN, EXPOSE, CMD) and their required order.
def generate_dockerfile(model_filename, port=8000, python_version="3.10", extra_packages=None):
    """
    Generate a Dockerfile string for serving a model.

    Args:
        model_filename (str): filename of the model artifact (e.g. 'iris_rf.joblib')
        port (int): port the app listens on
        python_version (str): Python version for the base image
        extra_packages (list): additional pip packages to install, or None

    Returns:
        str: complete Dockerfile content
    """
    # YOUR CODE HERE
    pass

# Two configurations prove your function handles ports, versions, and extras.
# --- Generate and print example Dockerfiles ---
df1 = generate_dockerfile("iris_rf.joblib", port=8000)
print("=== Dockerfile (basic) ===")
print(df1)

df2 = generate_dockerfile("sentiment_model.joblib", port=5000,
                           python_version="3.11",
                           extra_packages=["torch", "transformers"])
print("\n=== Dockerfile (with extra packages) ===")
print(df2)

In [ ]:
# --- Validation ---
assert df1 is not None, "generate_dockerfile must return a string"
assert isinstance(df1, str), "Result must be a string"

# Required instructions
for instruction in ["FROM python:3.10-slim", "WORKDIR /app",
                    "COPY requirements.txt", "iris_rf.joblib",
                    "EXPOSE 8000", 'CMD ["python", "app.py"]']:
    assert instruction in df1, f"Dockerfile missing: {instruction}"

# Extra packages
assert "torch" in df2 and "transformers" in df2, "Extra packages not in Dockerfile"
assert "3.11" in df2, "Python version not applied to base image"

print("Task 1 PASSED")

---
## Task 2 — Dockerfile Parser and Validator (25 points)

Implement `parse_dockerfile(dockerfile_str)` that returns a dict counting instructions:
```python
{"FROM": 1, "WORKDIR": 1, "COPY": 2, "RUN": 1, "EXPOSE": 1, "CMD": 1}
```

Then implement `validate_dockerfile(dockerfile_str)` that returns a list of issues (empty list = valid):
- Must have exactly 1 `FROM`
- Must have exactly 1 `WORKDIR`
- Must have exactly 1 `CMD`
- Must have at least 1 `COPY`
- Must have at least 1 `RUN`
- Must have exactly 1 `EXPOSE`

In [ ]:
# WHAT: Task 2 — implement a Dockerfile parser and a validator on top of it.
# WHY: CI systems lint Dockerfiles exactly like this — count instructions,
# then apply rules (exactly one FROM, a CMD present, WORKDIR set).
def parse_dockerfile(dockerfile_str):
    """
    Count instruction types in a Dockerfile string.
    Returns dict: {instruction_name: count}
    Only count these instructions: FROM, WORKDIR, COPY, RUN, EXPOSE, CMD
    """
    tracked = ["FROM", "WORKDIR", "COPY", "RUN", "EXPOSE", "CMD"]
    # Hint: split by lines, strip whitespace, check if line starts with each instruction
    # YOUR CODE HERE
    pass

def validate_dockerfile(dockerfile_str):
    """
    Validate a Dockerfile and return a list of issue strings.
    Returns empty list if valid.
    """
    # Hint: use parse_dockerfile, then check each rule
    # YOUR CODE HERE
    pass

# --- Test on Task 1 output ---
counts1 = parse_dockerfile(df1)
print("Instruction counts (basic Dockerfile):")
for instr, count in counts1.items():
    print(f"  {instr}: {count}")

issues1 = validate_dockerfile(df1)
if issues1:
    print(f"\nValidation issues: {issues1}")
else:
    print("\nValidation: No issues found")

# The validator must name what is MISSING, not just say 'invalid'.
# Test on a broken Dockerfile
broken_df = "COPY app.py /app/\nRUN pip install flask\n"
issues_broken = validate_dockerfile(broken_df)
print(f"\nBroken Dockerfile issues: {issues_broken}")

# Validation
assert counts1 is not None and isinstance(counts1, dict)
assert counts1.get("FROM") == 1, f"Expected 1 FROM, got {counts1.get('FROM')}"
assert counts1.get("CMD") == 1, f"Expected 1 CMD, got {counts1.get('CMD')}"
assert issues1 == [], f"Valid Dockerfile should have no issues, got: {issues1}"
assert len(issues_broken) >= 2, f"Broken Dockerfile should have ≥2 issues (missing FROM, WORKDIR, CMD)"
print("\nTask 2 PASSED")

---
## Task 3 — Container Isolation Simulation (25 points)

A key Docker property is **isolation**: two containers with conflicting library versions can run on the same server without interfering.

Simulate two containers as Python dicts with:
- `name`: container name
- `image`: base image string
- `env`: dict of environment variables
- `packages`: dict of `{package: version}`
- `port`: internal port the service listens on
- `host_port`: the port mapped on the host machine

Implement:
1. `create_container(name, image, env, packages, port, host_port)` → dict
2. `check_port_conflict(containers)` → list of conflicting `host_port` values (if two containers map to the same host port)

In [ ]:
# WHAT: Task 3 — model containers as config dicts and detect host-port conflicts.
# WHY: two containers can both use internal port 8000 — the conflict only
# exists on HOST ports; this simulation is exactly Docker's isolation story.
def create_container(name, image, env, packages, port, host_port):
    """
    Create a container configuration dict.
    Returns: {name, image, env, packages, port, host_port}
    """
    # YOUR CODE HERE
    pass

def check_port_conflict(containers):
    """
    Check if any two containers in the list share the same host_port.
    Returns list of conflicting host_port values (empty = no conflicts).
    """
    # Hint: collect all host_ports, find duplicates
    # YOUR CODE HERE
    pass

# c1 and c2 coexist (different host ports); c3 collides with c1 on 8001.
# --- Create two containers with different sklearn versions ---
c1 = create_container(
    name="iris-api-v1",
    image="python:3.10-slim",
    env={"MODEL_VERSION": "1.0", "PORT": "8000"},
    packages={"scikit-learn": "1.3.0", "flask": "3.0.0", "joblib": "1.3.2"},
    port=8000,
    host_port=8001  # mapped on the host
)

c2 = create_container(
    name="iris-api-v2",
    image="python:3.11-slim",
    env={"MODEL_VERSION": "2.0", "PORT": "8000"},
    packages={"scikit-learn": "1.4.0", "flask": "3.0.0", "joblib": "1.3.2"},
    port=8000,
    host_port=8002  # different host port — no conflict
)

c3 = create_container(
    name="iris-api-broken",
    image="python:3.9-slim",
    env={"MODEL_VERSION": "1.5"},
    packages={"scikit-learn": "1.2.0"},
    port=8000,
    host_port=8001  # CONFLICT with c1!
)

print("Container 1:", json.dumps(c1, indent=2))
print("\nContainer 2 sklearn version:", c2["packages"]["scikit-learn"])
print("Both containers use internal port 8000, mapped to different host ports.")
print("This is how Docker enables isolation.")

conflicts_ok = check_port_conflict([c1, c2])
conflicts_bad = check_port_conflict([c1, c2, c3])
print(f"\nConflicts (c1, c2)        : {conflicts_ok}  ← expected []") 
print(f"Conflicts (c1, c2, c3)    : {conflicts_bad}  ← expected [8001]")

# Validation
assert c1 is not None and isinstance(c1, dict)
assert c1["name"] == "iris-api-v1"
assert c1["packages"]["scikit-learn"] == "1.3.0"
assert c2["packages"]["scikit-learn"] == "1.4.0", "Containers must have independent package dicts"
assert conflicts_ok == [], f"No conflict expected, got {conflicts_ok}"
assert 8001 in conflicts_bad, f"Port 8001 conflict expected, got {conflicts_bad}"
print("\nTask 3 PASSED")

---
## Task 4 — Complete Deployment File Bundle (20 points)

A real containerized deployment needs 4 files written to disk:
1. `Dockerfile` — generated using Task 1
2. `requirements.txt` — pinned dependency versions
3. `app.py` — a minimal Flask app that loads the iris model and serves `/predict`
4. `README.txt` — build and run instructions

Write all 4 files to `EXERCISE_DIR` and verify they exist.

In [ ]:
# --- Task 4: Write all 4 deployment files ---

# TODO 4a: Write Dockerfile using generate_dockerfile()
#   model: "iris_rf.joblib", port: 8000
dockerfile_content = generate_dockerfile("iris_rf.joblib", port=8000)
# YOUR CODE: write to os.path.join(EXERCISE_DIR, "Dockerfile")

# TODO 4b: Write requirements.txt with these exact dependencies:
#   flask==3.0.0
#   scikit-learn==1.4.0
#   joblib==1.3.2
#   numpy==1.26.0
requirements = "flask==3.0.0\nscikit-learn==1.4.0\njoblib==1.3.2\nnumpy==1.26.0\n"
# YOUR CODE: write to os.path.join(EXERCISE_DIR, "requirements.txt")

# TODO 4c: Write app.py — a minimal Flask app:
#   - loads "/app/iris_rf.joblib" at startup
#   - has /predict POST endpoint (same as example notebooks)
#   - has /health GET endpoint
app_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
model = joblib.load("/app/iris_rf.joblib")
CLASS_NAMES = ["setosa", "versicolor", "virginica"]

@app.get("/health")
def health():
    return jsonify({"status": "ok", "model_loaded": True})

@app.post("/predict")
def predict():
    data = request.get_json(silent=True)
    if not data:
        return jsonify({"error": "JSON body required"}), 400
    try:
        features = [float(data[k]) for k in
                    ["sepal_length", "sepal_width", "petal_length", "petal_width"]]
    except (KeyError, TypeError, ValueError) as e:
        return jsonify({"error": str(e)}), 422
    arr = np.array([features])
    proba = model.predict_proba(arr)[0]
    idx = int(np.argmax(proba))
    return jsonify({"prediction": CLASS_NAMES[idx], "confidence": round(float(proba[idx]), 4)})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000)
'''
# YOUR CODE: write app_code to os.path.join(EXERCISE_DIR, "app.py")

# TODO 4d: Write README.txt with build and run instructions
readme = """
Iris Classifier — Docker Deployment
====================================
Build the image:
  docker build -t iris-api:1.0 .

Run the container:
  docker run -p 8000:8000 iris-api:1.0

Test the API:
  curl -X POST http://localhost:8000/predict \\
       -H 'Content-Type: application/json' \\
       -d '{"sepal_length":5.1,"sepal_width":3.5,"petal_length":1.4,"petal_width":0.2}'

Health check:
  curl http://localhost:8000/health
"""
# YOUR CODE: write readme to os.path.join(EXERCISE_DIR, "README.txt")

# --- Verify all 4 files exist ---
expected_files = ["Dockerfile", "requirements.txt", "app.py", "README.txt"]
print("Deployment bundle contents:")
for filename in expected_files:
    path = os.path.join(EXERCISE_DIR, filename)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  [{'OK' if exists else 'MISSING'}] {filename} ({size} bytes)")
    assert exists, f"{filename} was not written to {EXERCISE_DIR}"

print(f"\nAll deployment files written to: {EXERCISE_DIR}")
print("Task 4 PASSED")

In [ ]:
# WHAT: the final gate — one PASS/FAIL line per task.
# WHY: like a CI pipeline, the lab is done only when every gate is green;
# a FAIL points to the exact task to revisit.
# --- Final summary ---
print("=" * 55)
print("UNIT 4 LAB — FINAL GATE")
print("=" * 55)

gate = {
    "Task 1: Dockerfile generator produces valid content": df1 is not None and "FROM" in df1,
    "Task 2: Parser counts instructions correctly": counts1.get("FROM") == 1 if counts1 else False,
    "Task 3: Isolation simulation detects port conflicts": 8001 in (conflicts_bad or []),
    "Task 4: All 4 deployment files written": all(
        os.path.exists(os.path.join(EXERCISE_DIR, f))
        for f in ["Dockerfile", "requirements.txt", "app.py", "README.txt"]
    ),
}
for task, ok in gate.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {task}")
print("=" * 55)

---
## Self-Check Questions

1. **Why use `python:3.10-slim` instead of `python:3.10`?** How does the slim variant differ, and what is the trade-off?
2. **Why copy `requirements.txt` before copying the app code?** How does Docker's layer caching make this order faster?
3. **Two containers both use `scikit-learn` but different versions. Can they run on the same server without interfering?** Why?
4. **The `EXPOSE 8000` instruction does NOT open port 8000 to the internet.** What does it actually do, and what command maps it to the host?
5. **In Task 4, the `app.py` loads the model at startup (`model = joblib.load(...)`). What would happen to latency if the model loaded inside the `/predict` function instead?**